In [ ]:
%load_ext autoreload
%autoreload 2

## Imports

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import flax

In [ ]:
do_64_bit = False
if do_64_bit: jax.config.update("jax_enable_x64", True)

In [ ]:
import solve
import aux_ as aux
import RANK

## Device

In [10]:
device = aux.choose_gpu()

choosing cpu


## Training the neural network to the DSS and SSS

In [ ]:
model = RANK.RANK_model(device)

In [ ]:
sigma_quad = {
    "sigma_eps_u" : 0.0082,
    "sigma_eps_z" : 0.0044,
    "sigma_eps_Gamma" : 0.0097
}

sigma_sim = { # HIGHER?
    "sigma_eps_u" : 0.0082*2,
    "sigma_eps_z" : 0.0044*2,
    "sigma_eps_Gamma" : 0.0097*2
}

episode_list = [50_000, 1_000_000, 5_000_000, 1_000_000]
lr_list = [1e-4, 1e-4, 5e-5, 1e-6]
N_list = [500, 500, 1000, 1000]
ZLB_list = [-1, 0.00, 0.00, 0.00]

In [ ]:
zero_var_list = [None, None, None, None]
solve.train_nn(model, episode_list, sigma_sim, sigma_quad, ZLB_list, lr_list, N_list, zero_var_list, print_freq=100)

# ERGODIC DISTRIBUTION

In [ ]:
import jax
from neural_nets import eval_nn
import linear

In [ ]:
Y, pi = eval_nn(model.par, model.train, model.linear, model.nn, states[0], N)

Y_extra, pi_extra = eval_nn(model.par, model.train, model.linear, model.train["nn_phase0"], states[0], N)

In [ ]:
# maximum observations
data_max = max(model.sim.Y_OccBin.max(), model.sim.Y_lin.max())
sh_bins = np.linspace(model.sim.Y_lin.min(), data_max, 101)

f, ax = plt.subplots(1,3, figsize=(15,5))
ax[0].hist(model.sim.Y_OccBin, bins=sh_bins, alpha=0.5, label='OccBin', density=True)
ax[0].hist(model.sim.Y_lin, bins=sh_bins, alpha=0.5, label='Linear', density=True)
ax[0].set_title(r'Output: $Y_t$')

data_max = max(model.sim.pi_OccBin.max(), model.sim.pi_lin.max())
sh_bins = np.linspace(model.sim.pi_lin.min(), data_max, 101)

ax[1].hist(model.sim.pi_OccBin, bins=sh_bins, alpha=0.5, label='OccBin')
ax[1].hist(model.sim.pi_lin, bins=sh_bins, alpha=0.5, label='Linear')
ax[1].set_title(r'Inflation: $\pi_t$')

data_max = max(model.sim.i_OccBin.max(), model.sim.i_lin.max())
sh_bins = np.linspace(model.sim.i_lin.min(), data_max, 101)

ax[2].hist(model.sim.i_OccBin, bins=sh_bins, alpha=0.5, label='OccBin')
ax[2].hist(model.sim.i_lin, bins=sh_bins, alpha=0.5, label='Linear')
ax[2].set_title(r'Nominal Interest Rate: $i_t$')

ax[2].legend()

f.tight_layout()

#f.savefig('Ergodic_distribution_lin_OccBin.png')

In [ ]:
import pickle

with open('info_train.pkl', 'wb') as f:
    pickle.dump(model.info, f)